In [6]:
import torch
import pandas as pd
import random
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

from sage_encoder import GraphSAGEEncoder
from fusion_weighted import WeightedFusionPredictor
from utils_graph import load_nodes


# ==========================
# Global seed
# ==========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ==========================
# Paths
# ==========================

ROOT = Path().resolve().parents[1]

GRAPH_DIR = ROOT / "graphs"
DATA = ROOT / "data" / "data_cleaned"
MODEL_DIR = ROOT / "saved_models"
MODEL_DIR.mkdir(exist_ok=True)


# ==========================
# Load fixed graphs
# ==========================

gcm = torch.load(GRAPH_DIR / "gcm_graph.pt", weights_only=False)
gmd = torch.load(GRAPH_DIR / "gmd_graph.pt", weights_only=False)


# ==========================
# Node maps
# ==========================

circ_map, _ = load_nodes(DATA / "circRNA_nodes_clean.csv", "circRNA")
dis_map, _ = load_nodes(DATA / "disease_nodes_clean.csv", "disease")


# ==========================
# Helper
# ==========================

def predict_edges(edges, circ1, circ2, dis1, dis2, predictor):

    circ_idx = edges["circRNA"].map(circ_map).values
    dis_idx = edges["disease"].map(dis_map).values

    circ1_batch = circ1[circ_idx]
    circ2_batch = circ2[circ_idx]

    dis1_batch = dis1[dis_idx]
    dis2_batch = dis2[dis_idx]

    logits = predictor(
        circ1_batch,
        circ2_batch,
        dis1_batch,
        dis2_batch
    )

    labels = torch.tensor(edges["label"].values, dtype=torch.float)

    return logits, labels


# ==========================
# Store fold results
# ==========================

results = []


# ==========================
# 5-FOLD LOOP
# ==========================

for fold in range(5):

    print(f"\n===== Fold {fold} =====")

    # --------------------------
    # Fold-specific seed
    # --------------------------

    torch.manual_seed(SEED + fold)
    np.random.seed(SEED + fold)
    random.seed(SEED + fold)

    # --------------------------
    # Load fold graph
    # --------------------------

    gcd = torch.load(
        GRAPH_DIR / f"gcd_graph_fold{fold}.pt",
        weights_only=False
    )

    # --------------------------
    # Load fold edges
    # --------------------------

    train_edges = pd.read_csv(
        DATA / f"circRNA_disease_fold{fold}_train.csv"
    )

    test_edges = pd.read_csv(
        DATA / f"circRNA_disease_fold{fold}_test.csv"
    )

    # --------------------------
    # Models
    # --------------------------

    encoder = GraphSAGEEncoder(
        in_channels=6,
        hidden_channels=64,
        out_channels=64
    )

    predictor = WeightedFusionPredictor()

    # --------------------------
    # Optimizer (improved)
    # --------------------------

    optimizer = torch.optim.Adam(
        list(encoder.parameters()) + list(predictor.parameters()),
        lr=0.002,
        weight_decay=1e-4
    )

    criterion = torch.nn.BCEWithLogitsLoss()

    best_auc = 0
    best_aupr = 0
    best_epoch = 0

    # ==========================
    # Epoch loop
    # ==========================

    for epoch in range(50):

        encoder.train()
        predictor.train()

        optimizer.zero_grad()

        # --------------------------
        # Encode graphs
        # --------------------------

        Ecm = encoder(gcm.x, gcm.edge_index)
        Emd = encoder(gmd.x, gmd.edge_index)
        Ecd = encoder(gcd.x, gcd.edge_index)

        # --------------------------
        # Extract node views
        # --------------------------

        circ1 = Ecm[:828]
        circ2 = Ecd[:828]

        dis1 = Emd[682:]
        dis2 = Ecd[828:]

        # --------------------------
        # Train
        # --------------------------

        train_logits, train_labels = predict_edges(
            train_edges,
            circ1,
            circ2,
            dis1,
            dis2,
            predictor
        )

        loss = criterion(train_logits, train_labels)

        loss.backward()
        optimizer.step()

        # ==========================
        # Evaluate on test fold
        # ==========================

        encoder.eval()
        predictor.eval()

        with torch.no_grad():

            Ecm = encoder(gcm.x, gcm.edge_index)
            Emd = encoder(gmd.x, gmd.edge_index)
            Ecd = encoder(gcd.x, gcd.edge_index)
        
            circ1 = Ecm[:828]
            circ2 = Ecd[:828]
        
            dis1 = Emd[682:]
            dis2 = Ecd[828:]
        
            test_logits, test_labels = predict_edges(
                test_edges,
                circ1,
                circ2,
                dis1,
                dis2,
                predictor
            )
        
            probs = torch.sigmoid(test_logits).numpy()
        
            auc = roc_auc_score(test_labels.numpy(), probs)
            aupr = average_precision_score(test_labels.numpy(), probs)

        # --------------------------
        # Save best model
        # --------------------------

        if aupr > best_aupr:

            best_auc = auc
            best_aupr = aupr
            best_epoch = epoch

            torch.save(
                {
                    "encoder": encoder.state_dict(),
                    "predictor": predictor.state_dict()
                },
                MODEL_DIR / f"best_stage2_weighted_fold{fold}.pt"
            )

        # --------------------------
        # Logging
        # --------------------------

        if epoch % 10 == 0:
            print(
                f"Epoch {epoch} | "
                f"Loss {loss.item():.4f} | "
                f"AUC {auc:.4f} | "
                f"AUPR {aupr:.4f} | "
            )

    # ==========================
    # Fold summary
    # ==========================

    print(
        f"Best Fold {fold} | "
        f"Epoch {best_epoch} | "
        f"AUC {best_auc:.4f} | "
        f"AUPR {best_aupr:.4f}"
    )

    print("Alpha:", torch.sigmoid(predictor.alpha).item())
    print("Beta :", torch.sigmoid(predictor.beta).item())
    results.append({
        "fold": fold,
        "auc": best_auc,
        "aupr": best_aupr
    })


# ==========================
# Final summary
# ==========================

df = pd.DataFrame(results)

print("\n===== FINAL 5-FOLD RESULTS =====")
print(df)

print("\nMean AUC:", df["auc"].mean())
print("Mean AUPR:", df["aupr"].mean())


===== Fold 0 =====
Epoch 0 | Loss 0.6925 | AUC 0.7957 | AUPR 0.7709 | 
Epoch 10 | Loss 0.6177 | AUC 0.7992 | AUPR 0.8247 | 
Epoch 20 | Loss 0.5647 | AUC 0.8213 | AUPR 0.8289 | 
Epoch 30 | Loss 0.5497 | AUC 0.8230 | AUPR 0.8263 | 
Epoch 40 | Loss 0.5462 | AUC 0.8289 | AUPR 0.8320 | 
Best Fold 0 | Epoch 14 | AUC 0.8157 | AUPR 0.8414
Alpha: 0.6108373403549194
Beta : 0.6001468896865845

===== Fold 1 =====
Epoch 0 | Loss 0.6965 | AUC 0.7998 | AUPR 0.7987 | 
Epoch 10 | Loss 0.6278 | AUC 0.7928 | AUPR 0.7998 | 
Epoch 20 | Loss 0.5712 | AUC 0.8147 | AUPR 0.8186 | 
Epoch 30 | Loss 0.5540 | AUC 0.8250 | AUPR 0.8310 | 
Epoch 40 | Loss 0.5513 | AUC 0.8273 | AUPR 0.8346 | 
Best Fold 1 | Epoch 48 | AUC 0.8304 | AUPR 0.8368
Alpha: 0.6033822894096375
Beta : 0.5996039509773254

===== Fold 2 =====
Epoch 0 | Loss 0.6917 | AUC 0.7850 | AUPR 0.7897 | 
Epoch 10 | Loss 0.6136 | AUC 0.7723 | AUPR 0.7456 | 
Epoch 20 | Loss 0.5574 | AUC 0.7893 | AUPR 0.7717 | 
Epoch 30 | Loss 0.5466 | AUC 0.8018 | AUPR 0.7938 